In [ ]:
import os
import shutil
from edgar import set_identity, Company
from rich.console import Console
import pandas as pd

set_identity("user@example.com")

DATA_DIR = os.path.join(os.getcwd(), "data", "income_statements")
tickers = ["AMZN", "GOOG", "META"]
console = Console()

In [ ]:
for ticker in tickers:
    ticker_dir = os.path.join(DATA_DIR, ticker)
    if os.path.exists(ticker_dir):
        shutil.rmtree(ticker_dir)
    os.makedirs(ticker_dir)

    company = Company(ticker)
    filings = company.get_filings(form="10-K")
    filings_df = filings.to_pandas().head(1)

    for _, row in filings_df.iterrows():
        filing_date = str(row["filing_date"])
        period = str(row["reportDate"])
        accession = row["accession_number"]

        console.rule(f"[bold]{ticker} — Period: {period} (Filed: {filing_date})")

        filing = company.get_filings(form="10-K").get(accession)
        xbrl = filing.xbrl()
        income = xbrl.statements.income_statement(view="detailed")
        console.print(income.render())

        income_df: pd.DataFrame = income.to_dataframe()
        fy_cols = [c for c in income_df.columns if "(FY)" in c]
        income_df = income_df[["label"] + fy_cols]

        # Convert to raw numbers (no formatting) for JSON output
        numeric_df = income_df.copy()
        for col in fy_cols:
            numeric_df[col] = pd.to_numeric(income_df[col], errors="coerce")

        # Add YoY pct change columns (most recent vs prior year)
        if len(fy_cols) >= 2:
            current_col = fy_cols[0]
            prior_col = fy_cols[1]
            yoy_label = f"YoY% ({current_col.split('(')[0].strip()} vs {prior_col.split('(')[0].strip()})"
            numeric_df[yoy_label] = (
                (numeric_df[current_col] - numeric_df[prior_col]) / numeric_df[prior_col].abs()
            ).apply(lambda x: round(x * 100, 2) if pd.notna(x) else None)

            if len(fy_cols) >= 3:
                prior2_col = fy_cols[2]
                yoy2_label = f"YoY% ({prior_col.split('(')[0].strip()} vs {prior2_col.split('(')[0].strip()})"
                numeric_df[yoy2_label] = (
                    (numeric_df[prior_col] - numeric_df[prior2_col]) / numeric_df[prior2_col].abs()
                ).apply(lambda x: round(x * 100, 2) if pd.notna(x) else None)

        display(numeric_df)

        filepath = os.path.join(ticker_dir, f"income_statement_{ticker}.json")
        with open(filepath, "w") as f:
            f.write(numeric_df.to_json(orient="records", indent=2))
        print(f"-> saved {os.path.relpath(filepath)}")

    print()